# Session 9 — Designing, Implementing, and Reporting an Event Study

**Course: Event Studies in Finance & Economics**

*Mathis Mourey*

---

Sessions 1 through 8b developed the full toolkit: normal return models, aggregation, parametric and non-parametric tests, cross-sectional regressions, long-horizon methods, extensions for multi-country and intraday settings, the treatment of confounding events, the conditional framework, and the regression-based GARCH approach. This session integrates everything into a practical guide for conducting an event study from start to finish.

The session is organized as a sequential workflow. Each step is described, the relevant design choices are listed with their trade-offs, and a checklist is provided. We then execute the full workflow on a new application (the impact of FDA drug approval decisions on pharmaceutical firm stock prices) to demonstrate the process end to end. This application is chosen because it is clean (precise event dates, unambiguous information content, large sample of publicly traded firms), well-studied (the FDA literature is extensive), and pedagogically complementary to the M&A application of Session 6 and the OPEC application of Session 8b.

**References for this session:**

- Boehmer, E., Masumeci, J. and Poulsen, A.B. (1991). Event-Study Methodology Under Conditions of Event-Induced Variance. *Journal of Financial Economics*, 30(2), 253--272.
- Brown, S.J. and Warner, J.B. (1985). Using Daily Stock Returns. *Journal of Financial Economics*, 14(1), 3--31.
- Kothari, S.P. and Warner, J.B. (2007). Econometrics of Event Studies. In B.E. Eckbo (ed.), *Handbook of Empirical Corporate Finance*, Volume 1. Elsevier, Chapter 1.
- MacKinlay, A.C. (1997). Event Studies in Economics and Finance. *Journal of Economic Literature*, 35(1), 13--39.
- McWilliams, A. and Siegel, D. (1997). Event Studies in Management Research: Theoretical and Empirical Issues. *Academy of Management Journal*, 40(3), 626--657.

## 1. The Event Study Workflow

An event study proceeds through eight steps. Each step involves design choices that affect the validity and power of the study. The choices should be made before looking at the data (pre-registration, in the language of experimental research) to avoid data snooping.

**Step 1. Define the research question and hypothesis.** What is the event? What is the expected sign of the abnormal return? Is the hypothesis one-sided or two-sided? The research question determines the sample, the event window, and the test statistics.

**Step 2. Identify the events and construct the sample.** Which firms experienced the event? What is the event date? What are the inclusion and exclusion criteria? This step produces a list of (firm, event date) pairs.

**Step 3. Choose the event window and estimation window.** The event window determines what is measured; the estimation window determines how normal returns are estimated. The buffer between them prevents contamination.

**Step 4. Choose the normal return model.** The market model is the default. Multi-factor models (Fama-French, Carhart) reduce the variance of abnormal returns and are preferable when firm characteristics (size, value) are correlated with the event.

**Step 5. Estimate normal returns and compute abnormal returns.** This is the mechanical step: run the regressions, compute ARs and CARs.

**Step 6. Aggregate and test.** Compute the CAAR and apply the test statistics. Report at least one parametric test (BMP or KP-adjusted BMP) and one non-parametric test (Corrado rank or GRANK).

**Step 7. Cross-sectional analysis.** If the research question concerns the determinants of the abnormal return, regress firm-level CARs on explanatory variables (Session 6).

**Step 8. Robustness checks and reporting.** Vary the event window, the normal return model, the sample (with and without confounders), and the test statistic. Report all results transparently.

## 2. Application: FDA Drug Approval Decisions

### 2.1 Research Question

We ask: do pharmaceutical firms earn abnormal stock returns when the FDA announces approval of a new drug? The prediction is straightforward. A drug approval resolves substantial uncertainty about the firm's future revenue. If the market had not fully anticipated the approval, the announcement should produce a positive abnormal return. The magnitude should depend on the drug's expected revenue (blockbuster vs. niche), the probability of approval that the market had priced in before the decision, and the firm's pipeline concentration (whether the drug represents a large or small share of the firm's value).

### 2.2 Sample Construction

We select a sample of FDA New Drug Application (NDA) and Biologics License Application (BLA) approvals for publicly traded U.S. pharmaceutical firms between 2022 and 2024. The approval dates are publicly available from the FDA's website. We restrict the sample to firms listed on NYSE or NASDAQ with sufficient trading history for the estimation window.

### 2.3 Design Choices

We make the following choices before examining the data.

Event window: $[-1, +1]$ (primary), with $[-5, +5]$ as a robustness check. The narrow window captures the immediate market reaction; the wider window checks for pre-announcement leakage (FDA decisions are sometimes anticipated through advisory committee votes or clinical trial results).

Estimation window: 250 trading days ending 10 days before the event window.

Normal return model: market model (primary), with Fama-French three-factor as robustness.

Test statistics: BMP test and Corrado rank test (primary), with cross-sectional t-test and generalized sign test as robustness.

Confounding event screen: exclude firms with earnings announcements, M&A activity, or other material corporate news within the $[-1, +1]$ window.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# -- Step 2: Sample of FDA approvals --
fda_sample = pd.DataFrame({
    'ticker': ['LLY', 'ABBV', 'BMY', 'REGN', 'VRTX', 'MRNA', 'BIIB', 'JAZZ', 'IONS', 'SGEN'],
    'drug_name': ['Mounjaro', 'Rinvoq (new ind.)', 'Sotyktu', 'Dupixent (new ind.)',
                  'Trikafta (pediatric)', 'Spikevax (updated)', 'Leqembi',
                  'Xywav (pediatric)', 'Qalsody', 'Adcetris (new ind.)'],
    'approval_date': pd.to_datetime([
        '2022-05-13', '2023-03-16', '2022-09-09', '2023-03-28',
        '2023-04-20', '2023-09-11', '2023-01-06',
        '2022-09-01', '2023-04-25', '2023-04-13'
    ]),
    'indication': ['Diabetes', 'Atopic dermatitis', 'Psoriasis', 'COPD',
                   'Cystic fibrosis', 'COVID-19', 'Alzheimer's',
                   'Narcolepsy', 'ALS', 'Lymphoma'],
    'blockbuster': [True, True, False, True, True, True, True, False, False, False],
    'confounder_flag': [False, False, False, False, False, False, False, False, False, True],
    'confounder_note': ['', '', '', '', '', '', '', '', '',
                        'Seagen M&A rumors during window'],
})

print("FDA Approval Sample")
print("=" * 90)
print(fda_sample[['ticker', 'drug_name', 'approval_date', 'indication', 'blockbuster']].to_string(index=False))
print(f"\nSample: {len(fda_sample)} approvals, {fda_sample['blockbuster'].sum()} blockbusters")
print(f"Flagged confounders: {fda_sample['confounder_flag'].sum()}")

## 3. Steps 3-5: Estimation Window, Normal Returns, and Abnormal Returns

We now execute the mechanical steps: download data, estimate the market model for each firm, and compute abnormal returns. The code consolidates the procedures from Sessions 2 and 3 into a single pipeline.

In [ ]:
# -- Steps 3-5: Download, estimate, compute ARs --
market_ticker = '^GSPC'
start_date = '2020-06-01'
end_date = '2024-06-01'

data_mkt = yf.download(market_ticker, start=start_date, end=end_date, progress=False)
if isinstance(data_mkt.columns, pd.MultiIndex):
    data_mkt.columns = data_mkt.columns.get_level_values(0)
mkt_ret = data_mkt['Close'].pct_change().dropna()

all_tickers = fda_sample['ticker'].tolist()
data_firms = yf.download(all_tickers, start=start_date, end=end_date, progress=False)
if isinstance(data_firms.columns, pd.MultiIndex):
    firm_prices = data_firms['Close']
else:
    firm_prices = data_firms[['Close']].rename(columns={'Close': all_tickers[0]})
firm_ret = firm_prices.pct_change().dropna()

common = mkt_ret.index.intersection(firm_ret.index)
mkt_ret = mkt_ret.loc[common]
firm_ret = firm_ret.loc[common]
trading_days = mkt_ret.index

est_length = 250
buffer = 10
event_windows = [(-1, 1), (-5, 5)]
event_days_primary = list(range(-1, 2))

# Estimate and compute
ar_panel = pd.DataFrame(index=event_days_primary, dtype=float)
model_params = {}
car_results = []

for _, row in fda_sample.iterrows():
    tick = row['ticker']
    edate = row['approval_date']

    if tick not in firm_ret.columns:
        continue

    eidx = trading_days.get_indexer([edate], method='ffill')[0]
    ev_s = eidx - 1
    ev_e = eidx + 1
    est_e = ev_s - buffer - 1
    est_s = est_e - est_length + 1

    if est_s < 0 or ev_e >= len(trading_days):
        continue

    y_est = firm_ret.iloc[est_s:est_e+1][tick].values
    x_est = mkt_ret.iloc[est_s:est_e+1].values
    valid = ~(np.isnan(y_est) | np.isnan(x_est))
    X = sm.add_constant(x_est[valid])
    ols = sm.OLS(y_est[valid], X).fit()
    alpha, beta = ols.params
    sigma = np.sqrt(ols.mse_resid)

    # CARs for both windows
    cars_row = {'ticker': tick, 'drug': row['drug_name'],
                'blockbuster': row['blockbuster'], 'confounder': row['confounder_flag'],
                'alpha': alpha, 'beta': beta, 'sigma': sigma}

    for tau_a, tau_b in event_windows:
        s = eidx + tau_a
        e = eidx + tau_b
        if e >= len(trading_days):
            cars_row[f'CAR_[{tau_a:+d},{tau_b:+d}]'] = np.nan
            continue
        y_ev = firm_ret.iloc[s:e+1][tick].values
        x_ev = mkt_ret.iloc[s:e+1].values
        ar = y_ev - (alpha + beta * x_ev)
        cars_row[f'CAR_[{tau_a:+d},{tau_b:+d}]'] = np.nansum(ar)

    # Primary window AR panel
    y_ev = firm_ret.iloc[ev_s:ev_e+1][tick].values
    x_ev = mkt_ret.iloc[ev_s:ev_e+1].values
    ar_primary = y_ev - (alpha + beta * x_ev)
    ar_panel[tick] = ar_primary

    model_params[tick] = {'sigma': sigma, 'alpha': alpha, 'beta': beta,
                          'L': valid.sum()}
    car_results.append(cars_row)

car_df = pd.DataFrame(car_results)
N = ar_panel.shape[1]

print(f"Processed {N} firms")
print(f"\nCARs [-1,+1] (%):")
for _, r in car_df.iterrows():
    flag = ' *CONFOUNDER*' if r['confounder'] else ''
    bb = ' [blockbuster]' if r['blockbuster'] else ''
    print(f"  {r['ticker']:5s} {r['drug']:25s} CAR = {r['CAR_[-1,+1]']*100:+7.2f}%{bb}{flag}")
print(f"\nCAAR [-1,+1]: {car_df['CAR_[-1,+1]'].mean()*100:+.2f}%")

## 4. Step 6: Aggregation and Testing

We apply the full battery of tests from Sessions 4 and 5 to the FDA sample. The table below collects all seven test statistics for the primary $[-1, +1]$ window.

In [ ]:
# -- Step 6: Full test battery --

# Cross-sectional t-test
cars_primary = car_df['CAR_[-1,+1]'].dropna()
N_test = len(cars_primary)
caar = cars_primary.mean()
se_cs = cars_primary.std(ddof=1) / np.sqrt(N_test)
t_cs = caar / se_cs
p_cs = 2 * (1 - stats.t.cdf(abs(t_cs), df=N_test - 1))

# Patell test (simplified)
sar_panel = ar_panel.copy()
for tick in sar_panel.columns:
    sar_panel[tick] = sar_panel[tick] / model_params[tick]['sigma']
T_w = len(event_days_primary)
csar = sar_panel.sum(axis=0) / np.sqrt(T_w)
t_patell = np.sqrt(N_test) * csar.mean()
p_patell = 2 * (1 - stats.norm.cdf(abs(t_patell)))

# BMP test
csar_mean = csar.mean()
csar_std = csar.std(ddof=1)
t_bmp = csar_mean / (csar_std / np.sqrt(N_test))
p_bmp = 2 * (1 - stats.t.cdf(abs(t_bmp), df=N_test - 1))

# Sign test
n_pos = (cars_primary > 0).sum()
t_sign = (n_pos - 0.5 * N_test) / np.sqrt(0.25 * N_test)
p_sign = 2 * min(stats.binom.cdf(n_pos, N_test, 0.5),
                 1 - stats.binom.cdf(n_pos - 1, N_test, 0.5))
p_sign = min(p_sign, 1.0)

# Generalized sign test (use p_hat = 0.5 as proxy without full estimation window calc)
p_hat_est = 0.52  # typical for daily returns
t_gsign = (n_pos - N_test * p_hat_est) / np.sqrt(N_test * p_hat_est * (1 - p_hat_est))
p_gsign = 2 * (1 - stats.norm.cdf(abs(t_gsign)))

# Corrado rank test (simplified: rank within event window + estimation period)
# Using simplified cross-sectional rank approach
rank_vals = cars_primary.rank() / (N_test + 1) - 0.5
rank_mean = rank_vals.mean()
rank_std = rank_vals.std(ddof=0)
t_rank = rank_mean / (rank_std / np.sqrt(N_test)) if rank_std > 0 else 0
p_rank = 2 * (1 - stats.norm.cdf(abs(t_rank)))

print("Step 6: Test Results for FDA Approval Sample, Window [-1, +1]")
print("=" * 65)
print(f"  CAAR = {caar*100:+.2f}%, N = {N_test}, Positive = {n_pos}/{N_test}")
print()
print(f"  {'Test':25s} {'Statistic':>10s} {'p-value':>10s}")
print(f"  {'-'*50}")
tests = [
    ('Cross-Sectional t', t_cs, p_cs),
    ('Patell', t_patell, p_patell),
    ('BMP', t_bmp, p_bmp),
    ('Sign test', t_sign, p_sign),
    ('Generalized sign', t_gsign, p_gsign),
    ('Rank test', t_rank, p_rank),
]
for name, t_val, p_val in tests:
    stars = '***' if p_val < 0.01 else ('**' if p_val < 0.05 else ('*' if p_val < 0.10 else ''))
    print(f"  {name:25s} {t_val:+10.3f} {p_val:10.4f} {stars}")

## 5. Step 7: Cross-Sectional Regression

We test whether the magnitude of the announcement return depends on whether the approved drug is a blockbuster (expected peak sales exceeding $1 billion per year). The prediction is that blockbuster approvals generate larger CARs because they resolve more uncertainty about the firm's future cash flows.

$$
CAR_i = \gamma_0 + \gamma_1 D_{blockbuster,i} + \eta_i
$$

In [ ]:
# -- Step 7: Cross-sectional regression --
reg_data = car_df.dropna(subset=['CAR_[-1,+1]']).copy()
reg_data['CAR_pct'] = reg_data['CAR_[-1,+1]'] * 100
reg_data['bb_dummy'] = reg_data['blockbuster'].astype(int)

y = reg_data['CAR_pct']
X = sm.add_constant(reg_data[['bb_dummy']])
cs_reg = sm.OLS(y, X).fit(cov_type='HC1')

print("Step 7: Cross-Sectional Regression")
print("=" * 55)
print("Dependent variable: CAR [-1,+1] (%)")
print()
print(cs_reg.summary2().tables[1].to_string())
print(f"\nR-squared: {cs_reg.rsquared:.3f}")
print(f"\nInterpretation: blockbuster approvals are associated with a")
print(f"  {cs_reg.params['bb_dummy']:+.2f} percentage point {'higher' if cs_reg.params['bb_dummy'] > 0 else 'lower'} CAR")

## 6. Step 8: Robustness Checks

A credible event study should demonstrate stability of the results across the following dimensions.

**Alternative event window.** We re-estimate the CAAR and test statistics for $[-5, +5]$.

**Excluding confounders.** We drop the flagged observation and re-estimate.

**Sensitivity to individual observations.** We compute the leave-one-out CAAR: the CAAR when each firm is dropped in turn. If the CAAR changes sign or significance when a single firm is dropped, the result is fragile.

In [ ]:
# -- Step 8: Robustness --

print("Step 8: Robustness Checks")
print("=" * 65)

# 8a: Alternative window
cars_wide = car_df['CAR_[-5,+5]'].dropna()
caar_wide = cars_wide.mean()
se_wide = cars_wide.std(ddof=1) / np.sqrt(len(cars_wide))
t_wide = caar_wide / se_wide
p_wide = 2 * (1 - stats.t.cdf(abs(t_wide), df=len(cars_wide) - 1))
print(f"\n  (a) Alternative window [-5,+5]:")
print(f"      CAAR = {caar_wide*100:+.2f}%, t = {t_wide:.3f}, p = {p_wide:.4f}, N = {len(cars_wide)}")

# 8b: Excluding confounders
clean = car_df[~car_df['confounder']]['CAR_[-1,+1]'].dropna()
caar_clean = clean.mean()
se_clean = clean.std(ddof=1) / np.sqrt(len(clean))
t_clean = caar_clean / se_clean
p_clean = 2 * (1 - stats.t.cdf(abs(t_clean), df=len(clean) - 1))
print(f"\n  (b) Excluding confounders (N = {len(clean)}):")
print(f"      CAAR = {caar_clean*100:+.2f}%, t = {t_clean:.3f}, p = {p_clean:.4f}")

# 8c: Leave-one-out
print(f"\n  (c) Leave-one-out sensitivity:")
for tick in cars_primary.index:
    loo = cars_primary.drop(tick)
    loo_caar = loo.mean() * 100
    loo_t = loo.mean() / (loo.std(ddof=1) / np.sqrt(len(loo)))
    print(f"      Drop {car_df.loc[car_df['ticker']==tick, 'ticker'].values[0] if tick in car_df['ticker'].values else tick:5s}: "
          f"CAAR = {loo_caar:+.2f}%, t = {loo_t:+.3f}")

## 7. The Publication-Quality Figure

The standard event study figure consists of four panels, as established in Session 3. We produce it here for the FDA sample as the primary visual output of the study.

In [ ]:
# -- Four-panel figure --
agg = pd.DataFrame(index=event_days_primary)
agg['AAR'] = ar_panel.mean(axis=1)
agg['CAAR'] = agg['AAR'].cumsum()
agg['AAR_SE'] = ar_panel.std(axis=1, ddof=1) / np.sqrt(N)
agg['CAAR_SE'] = np.sqrt((agg['AAR_SE'] ** 2).cumsum())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: CAAR
ax = axes[0, 0]
ax.plot(event_days_primary, agg['CAAR'] * 100, 'o-', color='#1f78b4', linewidth=2.5, markersize=6)
ax.fill_between(event_days_primary,
                (agg['CAAR'] - 1.96 * agg['CAAR_SE']) * 100,
                (agg['CAAR'] + 1.96 * agg['CAAR_SE']) * 100,
                alpha=0.2, color='#1f78b4', label='95% CI')
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='red', linewidth=1.5, linestyle='--', alpha=0.6, label='Approval date')
ax.set_xlabel('Event Day ($\\tau$)')
ax.set_ylabel('CAAR (%)')
ax.set_title('Panel A: CAAR with 95% Confidence Band')
ax.legend(fontsize=9, frameon=True)
ax.grid(True, alpha=0.2)

# Panel B: AAR bars
ax = axes[0, 1]
colors = ['#d95f02' if v >= 0 else '#1f78b4' for v in agg['AAR']]
ax.bar(event_days_primary, agg['AAR'] * 100, color=colors, edgecolor='white', width=0.6)
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='red', linewidth=1.5, linestyle='--', alpha=0.6)
ax.set_xlabel('Event Day ($\\tau$)')
ax.set_ylabel('AAR (%)')
ax.set_title('Panel B: Average Abnormal Return')
ax.grid(True, alpha=0.2)

# Panel C: Individual CARs
ax = axes[1, 0]
for col in ar_panel.columns:
    ax.plot(event_days_primary, ar_panel[col].cumsum() * 100, 'o-', alpha=0.5, linewidth=1, markersize=4)
ax.plot(event_days_primary, agg['CAAR'] * 100, 'k-', linewidth=2.5, label='CAAR')
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='red', linewidth=1.5, linestyle='--', alpha=0.6)
ax.set_xlabel('Event Day ($\\tau$)')
ax.set_ylabel('CAR (%)')
ax.set_title('Panel C: Individual CARs')
ax.legend(fontsize=9, frameon=True)
ax.grid(True, alpha=0.2)

# Panel D: Cross-section of CARs
ax = axes[1, 1]
sorted_cars = car_df.sort_values('CAR_[-1,+1]')
colors_d = ['#d95f02' if v >= 0 else '#1f78b4' for v in sorted_cars['CAR_[-1,+1]']]
ax.barh(range(len(sorted_cars)), sorted_cars['CAR_[-1,+1]'] * 100,
        color=colors_d, edgecolor='white')
ax.set_yticks(range(len(sorted_cars)))
labels = [f"{r['ticker']} ({r['drug_name'][:15]})" for _, r in sorted_cars.iterrows()]
ax.set_yticklabels(labels, fontsize=8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('CAR [-1,+1] (%)')
ax.set_title('Panel D: Cross-Section of CARs')
ax.grid(True, alpha=0.2)

plt.suptitle('FDA Drug Approval Event Study', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. Reporting Checklist

The following checklist summarizes the minimum reporting standards for a publishable event study, drawing on the recommendations of MacKinlay (1997), McWilliams and Siegel (1997), and Kothari and Warner (2007).

**Sample.** Report the number of events, the sample period, the source of event dates, the inclusion and exclusion criteria, and the number of observations lost at each screening step (data availability, estimation window requirements, confounding events).

**Design.** State the event window, the estimation window, the buffer, and the normal return model. Justify each choice by reference to the research question and the event's information environment.

**Descriptive statistics.** Report summary statistics on the CARs: mean, median, standard deviation, minimum, maximum, fraction positive. These statistics characterize the cross-sectional distribution and alert the reader to outliers and skewness.

**Test statistics.** Report at least one parametric test (BMP or KP-adjusted BMP) and one non-parametric test (Corrado rank or GRANK). Report the test statistics and p-values for at least two event windows.

**Graphical output.** Include the four-panel figure (CAAR with confidence band, AAR bars, individual CAR spaghetti plot, cross-sectional distribution).

**Cross-sectional regression (if applicable).** Report OLS with HC1 standard errors, the R-squared, the number of observations, and the VIF for each regressor. Discuss potential endogeneity.

**Robustness.** Report results for alternative event windows, alternative normal return models, exclusion of confounders, and leave-one-out sensitivity. If the results change qualitatively, discuss why.

**Limitations.** Acknowledge the sample size, the potential for confounding events, the choice of normal return model, and the assumptions of the test statistics.

## 9. Summary and Preview of Session 10

This session demonstrated the full event study workflow from research question to publication-quality output. The FDA drug approval application illustrated each step: sample construction with confounder screening, estimation of normal returns and abnormal returns, aggregation with a full battery of parametric and non-parametric tests, cross-sectional regression of CARs on drug characteristics, and a comprehensive robustness protocol.

The reporting checklist in Section 8 provides a template that the researcher can follow for any event study application. The key message is that the credibility of an event study rests not on any single test statistic but on the consistency of results across multiple design choices.

Session 10, the final session of this course, is a capstone. It presents three complete case studies (an earnings announcement study, an M&A study, and a regulatory shock study), each implemented from start to finish using the toolkit developed across all nine preceding sessions. The session concludes with a summary of the full course and a curated reading list for further study.

**Additional references:**

- Binder, J.J. (1998). The Event Study Methodology Since 1969. *Review of Quantitative Finance and Accounting*, 11(2), 111--137.
- Campbell, C.J., Cowan, A.R. and Salotti, V. (2010). Multi-Country Event-Study Methods. *Journal of Banking and Finance*, 34(12), 3078--3090.
- Corrado, C.J. (2011). Event Studies: A Methodology Review. *Accounting and Finance*, 51(1), 207--234.
- Thompson, R. (1985). Conditioning the Return-Generating Process on Firm-Specific Events: A Discussion of Event Study Methods. *Journal of Financial and Quantitative Analysis*, 20(2), 151--168.